**/replace with your title/Lab 3: Point Cloud Data Processing**
---

Please refer to following resources:

Tutorials: https://www.open3d.org/docs/latest/tutorial

API Docs: https://www.open3d.org/docs/latest/python_api/

Collection of resources related to point cloud processing: https://github.com/mmolero/awesome-point-cloud-processing


**IMPORTANT NOTES**

1. For each question, you can interact with the  AI-TA by entering you doubts, questions etc in the text cell Chat with AI-TA. You can type in anything related to the question course to have your dialogue. However dont edit the header - chat with AI-TA. After entering your text -- click on the Check/Discuss with AI-TA button below it.

2. When your cells produce large outputs (for example Q4, Q5 etc), the colab environment is unable to extract the notebook to send to TA. For these cells, I have provided a clear_output button. So click that to clear the outputs, before interacting with the AI-TA.

3. For question 6 onwards, you will need a GPU. On top right (Where RAM/DISK are indicated), click the down arrow to select a run time with T4 GPU. Unfortunately, the free tier is a non-guaranteed resource and you might be able to use it only for about 2-hours/day. I would suggest that you switch to using T4 GPU - only for the last question.  The rest can be done using standard CPU. (Of course you can pay about. INR 1700 to get access for about 56 hours)

or if you have a nvidia GPU based workstation/laptop, you can run this notebook on it by setting up a jupyter server  on your local machine and then connecting to it via Local Runtime option (ask gemini for more details)

4. The answer cells have the characters hash hash ans - dont edit it as the client uses that as a marker to extract the answer and send to AI-TA tutor.

Happy solving.

**Instructions for Instructors**

Please see the https://github.com/amrutur/ai-ta/blob/main/README.md
for more information

In [1]:
#@title Please run this cell to install the client to access the AI-TA
AI_TA_URL="https://ai-ta-326056429620.asia-south1.run.app/"
course_id="cp260"
notebook_id = "HW3"
institution_id="IISc"
term_id = "2025-26"
!pip install colab-grading-client==1.0.25
import colab_graCopy of 3_Lab_CP260_2026_Ading_client as ta

In [2]:
# @title Please run this cell to authenticate yourself using your gmail credentials. A separate window will open for this. copy the token and paste in the text box below and press enter key to complete the authentication. Sometimes text box doesnt display - in which case rerun the cell.
session=ta.authenticate(AI_TA_URL)


Paste your token here: ··········
Authenticated as: Bharadwaj Amrutur (amrutur@gmail.com)


In [3]:
%%capture
# @title Please run this cell to install open3d and download a point cloud data set for this lab
!pip install open3d
import open3d as o3d
import numpy as np
import copy
import plotly.graph_objects as go
!wget https://raw.githubusercontent.com/PointCloudLibrary/pcl/master/test/bunny.pcd -O bunny.pcd

In [4]:
#@title Run this cell to visualize the bunny point cloud. You can use the mouse to interact with the point cloud like rotating it, checking the coordinates of a point, zooming in and out.
#The open3d visualizer doesnt work in colab
#Use the following instead
#Ref: https://medium.com/@sim30217/visualize-point-cloud-using-open3d-in-colab-a05b2f780a96
def show_cloud (cloud):
  points = np.asarray(cloud.points)
  colors = None
  if cloud.has_colors():
      colors = np.asarray(cloud.colors)
  elif cloud.has_normals():
      colors = (0.5, 0.5, 0.5) + np.asarray(cloud.normals) * 0.5
  else:
      cloud.paint_uniform_color((1.0, 0.0, 0.0))
      colors = np.asarray(cloud.colors)

  fig = go.Figure(
      data=[
          go.Scatter3d(
              x=points[:,0], y=points[:,1], z=points[:,2],
              mode='markers',
              marker=dict(size=1, color=colors)
          )
      ],
      layout=dict(
          scene=dict(
              xaxis=dict(visible=True),
              yaxis=dict(visible=True),
              zaxis=dict(visible=True)
          )
      )
  )
  fig.show()

def show_mesh(mesh):
  vert=np.asarray(mesh.vertices)
  tgl=np.asarray(mesh.triangles)
  fig=go.Figure(data=[
      go.Mesh3d(
          x=vert[:,0],
          y=vert[:,1],
          z=vert[:,2],
          i=tgl[:,0],
          j=tgl[:,1],
          k=tgl[:,2],
          color='lightpink',
          opacity=0.5)
      ])
  fig.show()

def get_bbox_edges(box):
    # corners is (8, 3)
    corners = np.asarray(box.get_box_points())
    # Get the indices that would sort the array
    # np.lexsort takes keys in reverse order of precedence.
    # So, to sort by column 1 (primary), then 0 (secondary), then 2 (tertiary):
    # keys = [corners[:, 2], corners[:, 0], corners[:, 1]]
    sorted_indices = np.lexsort((corners[:, 0], corners[:, 2], corners[:, 1]))

    # Apply the sorted indices to the original array
    sorted_corners = corners[sorted_indices]

    # This specific sequence of indices connects the corners into a box wireframe
    # We follow the path: bottom square -> top square -> vertical pillars
    indices = [
        0, 1, 3, 2, 0, # Bottom face
        4, 5, 7, 6, 4, # Top face
        5, 1, 0, 4, 6, 2, 3, 7 # Connecting vertical pillars
    ]

    # Extract the coordinates in the specified order
    edge_points = sorted_corners[indices]
    return edge_points

def show_bboxes(pcd, bboxes=[]):
    points = np.asarray(pcd.points)
    data = []

    # 1. Add the Point Cloud
    data.append(go.Scatter3d(
        x=points[:, 0], y=points[:, 1], z=points[:, 2],
        mode='markers',
        marker=dict(size=1, color=points[:, 2], colorscale='Viridis', opacity=0.4)
    ))

    # 2. Add Bounding Boxes (as wireframes)
    for box in bboxes:
        edge_points = get_bbox_edges(box)
        data.append(go.Scatter3d(
            x=edge_points[:, 0],
            y=edge_points[:, 1],
            z=edge_points[:, 2],
            mode='lines',  # Changed from 'markers' to 'lines'
            line=dict(color='red', width=4)
        ))

    fig = go.Figure(data=data)
    fig.update_layout(scene_aspectmode='data')
    fig.show()

bunny = o3d.io.read_point_cloud("bunny.pcd")
show_cloud(bunny)

---
**Q1**

Build a KD tree for the bunny point cloud using the appropriate function from open 3D

Color a small ball region of radius 0.025 around the bunny's mouth in blue.

##Ans

(Do not delete this cell. Write your answers in the following cells)

In [5]:

# <your code> Build the bunny_tree
bunny_tree = o3d.geometry.KDTreeFlann(bunny)

# <your code> Find the coordinate oa point  approximately in the mouth
# from the plot output of the previous cell.

mouth_point = np.array([-0.089,0.118,0.045])

# <your code> Find the set of points in a ball of radius 0.025 around the mouth
[k,idx,_]=bunny_tree.search_radius_vector_3d(mouth_point,0.025)

# <your code> Set their color as blue
np.asarray(bunny.colors)[idx[1:],:]=[0,0,1.]


show_cloud(bunny)

**Chat with TA**

<REPLACE this line: Note: if you are the instructor, you can replace this text with your comments/questions to the TA to help you formulate a better question or answer rubric or even scoring rubric. For example, Check the question and answer.  If you are a student, you can replace this text with your question/comment or request to TA to help you with answering/understanding the concepts behind this question: for example: help me with this question>

In [6]:
ta.show_teaching_assist_button(session,AI_TA_URL, 1, notebook_id, institution_id, term_id, course_id)

Button(button_style='info', description='Check/Help with question Q1!', layout=Layout(width='auto'), style=But…

Please wait, asking the AI TA ...
Formed the prompt and asking the Tutor now...


Of course! I'd be happy to take a look at your lab question and the provided answer. This looks like a great hands-on exercise for getting students familiar with Open3D and fundamental point cloud operations.

Here is my feedback:

### Feedback on the Question

The question is **excellent**.

*   **Clarity & Conciseness:** It's very clear and broken into two distinct, manageable parts. Students know exactly what to do: 1) build a KD-tree, and 2) use it to perform a radius search and modify the point cloud.
*   **Relevance:** This is highly relevant to the course material. The lectures discuss how KD-trees are used for efficient nearest neighbor searches, which is a foundational block for more advanced algorithms like LOAM and Fast-LIO (for finding correspondences). This question provides a direct, practical application of that theory.
*   **Engagement:** Asking students to visually pick a point (like the bunny's mouth) is a nice touch. It encourages them to interact with and inspect the 3D data, which is a valuable skill.

I have no suggestions for improving the question itself. It's well-formulated.

### Feedback on the Answer

The provided answer is very good and follows the correct logical steps. However, there is one small but important bug I'd like to point out.

**Analysis of the Answer Code:**

1.  `bunny_tree = o3d.geometry.KDTreeFlann(bunny)`
    *   **Correct.** This is the correct Open3D function to build a KD-tree from the point cloud.

2.  `mouth_point = np.array([-0.089,0.118,0.045])`
    *   **Correct.** This defines the center of the search region as requested.

3.  `[k,idx,_]=bunny_tree.search_radius_vector_3d(mouth_point,0.025)`
    *   **Correct.** This correctly performs a radius search using the KD-tree, finding all points within a 0.025 radius of `mouth_point`. The `idx` variable will contain the indices of these points.

4.  `np.asarray(bunny.colors)[idx[1:],:]=[0,0,1.]`
    *   **This line has a subtle error.** The code uses `idx[1:]`, which slices the list of found indices and **skips the first point**. The `search_radius_vector_3d` method returns all points within the given radius, and there is no reason to exclude the first one in the returned list. All points found by the search should be colored blue.

**Suggested Improvement:**

The fix is to simply use the entire list of indices.

**Corrected Code:**

```python
# The original bunny point cloud might not have colors. 
# Let's ensure the color buffer exists before we try to modify it.
if not bunny.has_colors():
    bunny.paint_uniform_color([0.5, 0.5, 0.5]) # Paint it grey to start

bunny_tree = o3d.geometry.KDTreeFlann(bunny)

mouth_point = np.array([-0.089,0.118,0.045])

# The search returns the number of points, the indices, and the squared distances
[k, idx, _] = bunny_tree.search_radius_vector_3d(mouth_point, 0.025)

# Convert the Open3D vector of indices to a NumPy array for easier indexing
indices_to_color = np.asarray(idx)

# Access the colors of the bunny cloud as a NumPy array and set the color for the found indices
np.asarray(bunny.colors)[indices_to_color, :] = [0, 0, 1.0] # Blue

show_cloud(bunny)
```

**Why the improved code is better:**

1.  **Correctness:** It uses `np.asarray(idx)` to get all the indices, ensuring that every point found within the radius is colored, which fixes the bug.
2.  **Robustness:**
    *   Explicitly converting `idx` to a NumPy array with `np.asarray(idx)` is slightly more robust and makes the code's intent clearer.
    *   I also added a check `if not bunny.has_colors():` at the start. While your `show_cloud` helper function handles this implicitly, making it explicit in the student's answer cell can be good practice. It prevents an error if a student were to try modifying colors on a point cloud without a color buffer.

Overall, this is a great exercise! With that one small correction to the answer key, it will be perfect. Keep up the great work on the lab materials

**Built in 3D Objects**

Open3D allows for creation of some basic datasets like sphere, cylinder etc. Look at the API reference for the details.

3D objects can be meshed or in point cloud form. The meshes can be triangular or polygons in general.

A triangle mesh has three vertices and three edges.

Usually the vertices are given as an array of nx3 - vertices

Any triangle is indicated as a triplet of indices (i,j,k) where vertices[i], vertices[j], vertices[k] form the triangle.

Same format is used to specify any polygonal mesh.

In [ ]:
#lets create a sphere of radius 2.0 using triangle meshes
sphere=o3d.geometry.TriangleMesh.create_sphere(radius=2.0)
show_mesh(sphere)

---
You can use your mouse pointer to interactively view the object from different perspectives.

---
**Q2**

Find the volume of the rendered sphere and compare with that of an ideal sphere.

Why is the volume less than that of ideal sphere?

How could you modify the code to make the calculated volume closer to the ideal volume? Explain why your modification works."

In [ ]:
## Ans
radius = 2.0
# <your code> to compare the volume of this structure with that of a ideal sphere
print(sphere.get_volume(),np.pi*4*radius**3/3)

Because the sphere is approximated with triangles, whose vertices are on the sphere ,but are inscribed within the sphere, some volume is left out in between the triange face and the actual sphere surface.

The resolution parameter in create_sphere API can be used to control this error:


In [ ]:
# Example of improved sphere
# The default resolution is 20, so any value > 20.
sphere_high_res = o3d.geometry.TriangleMesh.create_sphere(radius=2.0, resolution=50)

# Optional but good: Show that the volume is now closer
print(f"High-Res Mesh Volume: {sphere_high_res.get_volume()}")

**Chat with TA**

Check the question and answer

In [ ]:
ta.show_teaching_assist_button(session,AI_TA_URL, 2, notebook_id, institution_id, term_id, course_id)

---
We will now create a point cloud from this mesh representation.

Many possible ways to sample are there - we will use the following API and create a point cloud with 3000 points.

In [ ]:
sphere_cloud = sphere.sample_points_poisson_disk(3000)
show_cloud(sphere_cloud)

---
**Q3**

Mark all the points within 0.1 units of the great circle oriented with a normal of (0,1,1) as blue in color.

In [ ]:
## Ans

#make a copy of the sphere so that
#original data is not modified
sp=copy.deepcopy(sphere_cloud)

# make all points gray
gray_color = np.array([0.5, 0.5, 0.5])
blue_color = np.array([0,0,1])
sp.paint_uniform_color(gray_color)

#desired tolerance
tol=0.1

# Your code below

#Unit normal to the great circle
u=np.array([0,1.,1.])
u = u / np.linalg.norm(u)

print(f"Number points in the cloud={len(sp.points)}")

# Core logic
#for i in range(len(sp.points)):
#  if (abs(u.dot(sp.points[i]))< tol):
#    sp.colors[i]=np.array([0,0.,1.0])

#write using numpy methods for parallel processing
pc_points = np.asarray(sp.points)
pc_colors = np.asarray(sp.colors)

#point to plane distance
pc_ppd = np.abs(np.dot(pc_points,u))

#mark points which are within the  distance threshold
blue_mask = pc_ppd < tol
pc_colors[blue_mask] = blue_color

show_cloud(sp)

**Chat with TA**

Check question and answer

In [ ]:
ta.show_teaching_assist_button(session,AI_TA_URL, 3, notebook_id, institution_id, term_id, course_id)

---
For the next question we will use the room scene point cloud.

Run the next cell to load and render the points cloud.
Play around with the Point cloud to understand the scene.

**IMPORTANT NOTE**

Please clear the output of this cell by clicking on the clear output button before any further interactions with the AI TA (Whether for asking assistance from AI TA or submitting your notebook for evaluation). This because currently colab environment hangs when it tries to extract this notebook to send to AI TA, due to the data size of a large point cloud.

In [ ]:
scene_ply_data = o3d.data.DemoCropPointCloud()
scene = o3d.io.read_point_cloud(scene_ply_data.point_cloud_path)
scene_points = np.asarray(scene.points)
scene_colors=np.asarray(scene.colors)
show_cloud(scene)
ta.show_clear_output_button()


### ✅ Current cell output cleared!

If you have **large outputs** from other cells (3D visualizations, plots), you can:

1. **Clear all outputs**: Go to `Runtime > Restart and clear outputs`
2. **Clear specific cell**: Click the three dots on a cell output and select "Clear output"

**Large outputs can cause submission errors**, especially from:
- Open3D 3D visualizations
- Large matplotlib plots
- Interactive widgets
- Large data displays

After clearing, you can safely use `ask_assist()` or `submit_eval()`.
    

---
**Q4**

Segment the floor by coloring the points green.

State your logic briefly within comments in the answer cell below
including any assumptions.

(Hint:

Search the internet for some tutorials on the RANSAC: Random
Sample Consensus - or ask the tutor using the Chat cell below.

You can then use an appropriate open3d function that implements RANSAC based segmentation)

In [ ]:
## Ans

#make a copy of the scene to edit for this question
scene_segmented = copy.deepcopy(scene)


# Your assumption:
#
# From examining the scene we see that floor points have a y coordinate
# of about 2.4. We will use that as a signature for floor points, but use
# RANSAC to cleanly segement it out to remove noisy/outlier points.

# Your code

#green color
green_color = np.array([0,1.0,0])

#To segment the floor we will first find all the points whose
# y coordinate is at about 2.4 within a small tolerance.
y_coord = 2.4

# We will then use RANSAC to fit a plane equation to this subset
# We will then color all the points which fit to this plane within
# a tolerance.

# Our assumption is that the floor points are all at a constant 'y' height.

# an array to keep track of the indices of the points which are deemed as floor.
floor_idx=np.zeros((len(scene_points)),np.int32)

#The floor's y-coordinate has a value of about 2.4
#as seen in the picture above
#Hence crop out the points within +- 0.2 of this

crop_tol = 0.2
floor_mask = np.abs(scene_points[:,1]-y_coord) < crop_tol

floor_indices = np.where(floor_mask)[0]
floor_cloud = scene.select_by_index(floor_indices)

#get ransac to fit to a plane which has maximum number of points
#within +- segment_tol of it.

segment_tol = 0.015
floor_plane,floor_plane_indices=floor_cloud.segment_plane(distance_threshold=segment_tol,
                                                    ransac_n=3,
                                                    num_iterations=20)

#convert this indices on floor plane to indices on the scene's point cloud
scene_floor_indices = floor_indices[floor_plane_indices]

[a, b, c, d] = floor_plane
print(f"Plane equation: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")
print(f"Found {len(floor_plane_indices)} floor points.")

#color the floor points of the scene
segmented_scene_colors = np.asarray(scene_segmented.colors)
# Use the segmented floor indices to color the points in the original scene copy.
segmented_scene_colors[scene_floor_indices] = green_color # Color the floor green


show_cloud(scene_segmented)
ta.show_clear_output_button()

**Chat with TA**

Check the question and answer

In [ ]:
ta.show_teaching_assist_button(session,AI_TA_URL, 4, notebook_id, institution_id, term_id, course_id)

---
**Q5**

In this question we want to compare ICP based points cloud registration by using point-to-point and point-to-plane distances.

The first point cloud is scene. We will create a new point cloud, new scene,  by rotating  scene via an arbitrary transform.

You task is to use the ICP apis to recover the applied  transformation.

You should try ICP with both  point-to-point and point-to-plane metrics.

Compare the final fitness and inlier_rmse from both methods. Which one performed better?

Furthermore, compare the recovered transformation matrix from each method to the ground-truth transformation you applied. Which method recovered the original transformation more accurately and why?


In [ ]:
## Ans

def get_se3_transform(angle:float, axis:np.ndarray, translate: np.ndarray)->np.ndarray:
  '''
  return a SE3 4x4 transform given rotation angle (radians), rotation axis (3x1), translate vector (3x1)
  '''
  rot_m=o3d.geometry.get_rotation_matrix_from_axis_angle(angle*axis)
  trans=np.zeros((4,4))
  trans[:3,:3]=rot_m
  trans[:3,3]=translate.flatten()
  trans[3,3]= 1.0
  return trans

# This code sets up the transform to obtain new_scene from scene

#arbitrary axis, angle and translation values
ang=np.pi/4.0
axis=np.array([0,1.,1.0])
translate = np.array([0,0,0.3])

applied_trans = get_se3_transform(ang, axis, translate)
print(f"The applied transfomation is : {applied_trans}")

#create a new scene with the above transformation
new_scene=copy.deepcopy(scene)
new_scene.transform(applied_trans)
print(f"Number of points in point cloud is : {len(new_scene.points)}")

In [ ]:
# Your code below

# Find normals at each point in the original and new scene by examining a
# neighbourhood ball of radius 0.1 around the point. Cap the maximum number
# of nearest neighbours to be 30.

new_scene.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.1, max_nn=30))

scene.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.1, max_nn=30))

# Initialize the transform to be identity
trans_init = np.eye(4)

max_correspondence_distance = 0.5
reg_original = o3d.pipelines.registration.evaluate_registration(
        scene, new_scene, max_correspondence_distance, trans_init)

print(f"The unregistered inlier_rmse: {reg_original.inlier_rmse}")

#Find best registeation with point to plane metric
reg_p2p=o3d.pipelines.registration.registration_icp(
    source=scene,
    target=new_scene,
    init=trans_init,
    max_correspondence_distance=max_correspondence_distance,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    )
print(f"With point to plane, registation inlier rmse is: {reg_p2p.inlier_rmse}")
print(f"The estimated transform is {reg_p2p.transformation} and its error is :{np.linalg.norm(applied_trans-reg_p2p.transformation)}")

#Find best registration with point to distance metric
reg_p2l=o3d.pipelines.registration.registration_icp(
    source=scene,
    target=new_scene,
    init=trans_init,
    max_correspondence_distance=max_correspondence_distance,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    )
print(f"With point to point, registration inlier_rmse is: {reg_p2l.inlier_rmse}")
print(f"The estimated transform is {reg_p2l.transformation} and its error is :{np.linalg.norm(applied_trans-reg_p2l.transformation)}")


**Observations**

Before applying ICP, the registration is poor as expected, with a fitness of only 0.276 and a high inliner_rmse of about 0.3

Applying point to plane ICP dramatically improves the registration results, an inlier_rmse of almost 0. On the other hand, the point to point rmse, does reduce by a factor of 10, but is considerably higher than that of point to plane.

We see this impact in the accuracy of the estimated transform also, where the point-to-plane estimated transform has almost 0 error, while the point-to-point estimated transform has an arror of 0.2.

We believe that the point-to-plane is working much better, due to the presence of strong plane segments (floor and walls).The point-to-plane metric minimizes the distance from a source point to the tangent plane of the corresponding target point. In a scene with large flat areas (like walls and floors), this constraint is more effective than the point-to-point metric. It allows points to "slide" along the plane to find a better fit, converging faster and more accurately. The point-to-point metric can get stuck in local minima more easily, especially if the initial alignment is not very close.

**Chat with TA**

Check question and answer

In [ ]:
ta.show_teaching_assist_button(session,AI_TA_URL, 5, notebook_id, institution_id, term_id, course_id)

---
We will next look at Pointnext++ based inferencing and segmentation of the room scene. You can experiment with different scenes, changes to sampling method etc and see the results for yourself.

In [ ]:
!git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git
%cd Pointnet_Pointnet2_pytorch/

In [ ]:
# Copy the pre-trained classification model to your current folder
#!cp /content/Pointnet_Pointnet2_pytorch/log/classification/pointnet2_ssg_wo_normals/checkpoints/best_model.pth .
!cp ~/Pointnet_Pointnet2_pytorch/log/classification/pointnet2_ssg_wo_normals/checkpoints/best_model.pth .


In [ ]:
import torch
import numpy as np
import sys
import os
import torch.nn.functional as F

torch.serialization.add_safe_globals([np._core.multiarray.scalar])

# Assuming you are in /content/Pointnet_Pointnet2_pytorch
# This adds the 'models' directory to the Python path
sys.path.append(os.path.abspath('models'))

from models.pointnet2_cls_ssg import get_model

MODELNET40_CLASSES = {
    0: 'airplane', 1: 'bathtub', 2: 'bed', 3: 'bench', 4: 'bookshelf',
    5: 'bottle', 6: 'bowl', 7: 'car', 8: 'chair', 9: 'cone',
    10: 'cup', 11: 'curtain', 12: 'desk', 13: 'door', 14: 'dresser',
    15: 'flower_pot', 16: 'glass_box', 17: 'guitar', 18: 'keyboard', 19: 'lamp',
    20: 'laptop', 21: 'mantel', 22: 'monitor', 23: 'night_stand', 24: 'person',
    25: 'piano', 26: 'plant', 27: 'radio', 28: 'range_hood', 29: 'sink',
    30: 'sofa', 31: 'stairs', 32: 'stool', 33: 'table', 34: 'tent',
    35: 'toilet', 36: 'tv_stand', 37: 'vase', 38: 'wardrobe', 39: 'xbox'
}

#set the seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def farthest_point_sample(xyz, npoint):
    """
    Input:
        xyz: pointcloud data, [B, N, 3]
        npoint: number of samples
    Return:
        centroids: sampled pointcloud index, [B, npoint]
    """
    device = xyz.device
    B, N, C = xyz.shape
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, -1)
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, -1)[1]
    return centroids


def classify_point_cloud(point_cloud, checkpoint_path='best_model.pth'):
  # 1. Ensure exactly 1024 points (Sampling)
  points = point_cloud.copy() # NumPy array (N, 3)

  # 2. Normalize (Shift to origin and scale to unit sphere)
  points = points[:, 0:3]
  points -= np.mean(points, axis=0)
  points /= np.max(np.linalg.norm(points, axis=1))

  # Convert to float32 and then to PyTorch tensor
  # FIX 1: Use 'points' instead of undefined 'points_np'
  points_tensor_cpu = torch.from_numpy(points).float()
  # Add batch dimension: (N, 3) -> (1, N, 3)
  points_tensor_cpu = points_tensor_cpu.unsqueeze(0)

  # Move to CUDA
  points_gpu = points_tensor_cpu.cuda()
  #print(f"Torch tensor on GPU shape: {points_gpu.shape}, dtype: {points_gpu.dtype}")

  # Synchronize after moving to GPU (helps to get clearer error messages if previous ops were async)
  torch.cuda.synchronize()

  # Sample 1024 points using farthest point sampling
  sampled_point_indices = farthest_point_sample(points_gpu, 1024) # (1, 1024)
  #print(f"Sampled point indices shape: {sampled_point_indices.shape}")

  # Select the sampled points from the GPU tensor. Resulting shape (1, 1024, 3)
  # Then squeeze to (1024, 3) for further processing
  sampled_points_tensor = points_gpu[torch.arange(points_gpu.shape[0]).unsqueeze(-1), sampled_point_indices, :].squeeze(0)
  #print(f"Sampled points shape: {sampled_points_tensor.shape}")

  # Model expects input in (B, C, N) format for classification
  # sampled_points_tensor is currently (1024, 3)
  # We need to transform (1024, 3) -> (1, 3, 1024)

  # FIX 2: Correct reshape for model input
  points_tensor_model_input = sampled_points_tensor.unsqueeze(0).transpose(1, 2) # (1024, 3) -> (1, 1024, 3) -> (1, 3, 1024)

  # 4. Load Model and Weights
  model = get_model(num_class=40, normal_channel=False).cuda()
  checkpoint = torch.load(checkpoint_path, weights_only=False)
  model.load_state_dict(checkpoint['model_state_dict'])
  model.eval()

  # 5. Run Inference
  with torch.no_grad():
    logits, _ = model(points_tensor_model_input) # Use the correctly shaped tensor
    target_idx = logits.data.max(1)[1].item()
    # Apply Softmax to get probabilities (0.0 to 1.0)
    probs = F.softmax(logits, dim=1)
    # Convert to a readable numpy array
    probs_array = probs.cpu().numpy()[0]
    return MODELNET40_CLASSES[target_idx], probs_array


# Usage:

# Example Usage:
result, probabilities = classify_point_cloud(np.asarray(bunny.points))
print(f"Predicted class is : {result}")
top5_indices = probabilities.argsort()[-5:][::-1]

print("Top 5 Predictions:")
for idx in top5_indices:
    name = MODELNET40_CLASSES[idx]
    confidence = probabilities[idx] * 100
    print(f"{name:15} : {confidence:.2f}%")


In [ ]:
# Segment all the planes out

def segment_all_planes(pcd, max_planes=3, dist_thresh=0.02):
    segment_results = []
    current_pcd = pcd

    for i in range(max_planes):
        # 1. Find the largest plane
        plane_model, inliers = current_pcd.segment_plane(
            distance_threshold=dist_thresh, ransac_n=3, num_iterations=1000
        )

        # 2. Extract and save the plane
        plane_pcd = current_pcd.select_by_index(inliers)
        plane_pcd.paint_uniform_color(np.random.rand(3)) # Color for viz
        segment_results.append(plane_pcd)

        # 3. Keep only the "outliers" (non-plane points) for next iteration
        current_pcd = current_pcd.select_by_index(inliers, invert=True)

        if len(current_pcd.points) < 10: break

    return segment_results, current_pcd

planes, remaining_objects = segment_all_planes(scene)

#remaining objects has the left over point cloud after removal of planes

In [ ]:
#visualize the remaining objects after clustering them using dbscan


# labels is an array of integers where -1 is noise, 0 is Object A, 1 is Object B, etc.
labels = np.array(remaining_objects.cluster_dbscan(eps=0.05, min_points=10))

max_label = labels.max()
print(f"Detected {max_label + 1} distinct objects in the scene.")

# Color the clusters to visualize them
import matplotlib.pyplot as plt
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0 # Noise points are black
remaining_objects.colors = o3d.utility.Vector3dVector(colors[:, :3])

show_cloud(remaining_objects)
ta.show_clear_output_button()


### ✅ Current cell output cleared!

If you have **large outputs** from other cells (3D visualizations, plots), you can:

1. **Clear all outputs**: Go to `Runtime > Restart and clear outputs`
2. **Clear specific cell**: Click the three dots on a cell output and select "Clear output"

**Large outputs can cause submission errors**, especially from:
- Open3D 3D visualizations
- Large matplotlib plots
- Interactive widgets
- Large data displays

After clearing, you can safely use `ask_assist()` or `submit_eval()`.
    

In [ ]:
# List to store the boxes for visualization
bboxes = []
object_types = []
cluster_pcds = {}
for i in range(labels.max() + 1):
    # Extract points for this specific cluster
    cluster_indices = np.where(labels == i)[0]
    cluster_pcds[i] = remaining_objects.select_by_index(cluster_indices)

    # Calculate Oriented Bounding Box (Tight fit)
    obb = cluster_pcds[i].get_oriented_bounding_box()
    obb.color = (1, 0, 0) # Red for OBB

    # Calculate Axis-Aligned Bounding Box (Standard fit)
    aabb = cluster_pcds[i].get_axis_aligned_bounding_box()
    aabb.color = (0, 0, 1) # Blue for AABB

    #bboxes.append(obb)
    bboxes.append(aabb) # Uncomment to see both

    object_types.append(classify_point_cloud(np.asarray(cluster_pcds[i].points))[0])

show_bboxes(remaining_objects, bboxes)
ta.show_clear_output_button()


### ✅ Current cell output cleared!

If you have **large outputs** from other cells (3D visualizations, plots), you can:

1. **Clear all outputs**: Go to `Runtime > Restart and clear outputs`
2. **Clear specific cell**: Click the three dots on a cell output and select "Clear output"

**Large outputs can cause submission errors**, especially from:
- Open3D 3D visualizations
- Large matplotlib plots
- Interactive widgets
- Large data displays

After clearing, you can safely use `ask_assist()` or `submit_eval()`.
    

In [ ]:
print(object_types)

If you see in the above list, the chair doesnt appear.  However looks like the cluster labeled ' tent' seems to be the point cloud for chair (index 6).


In [ ]:
cluster_id = 6
show_bboxes(cluster_pcds[cluster_id], [bboxes[cluster_id]])
ta.show_clear_output_button()


### ✅ Current cell output cleared!

If you have **large outputs** from other cells (3D visualizations, plots), you can:

1. **Clear all outputs**: Go to `Runtime > Restart and clear outputs`
2. **Clear specific cell**: Click the three dots on a cell output and select "Clear output"

**Large outputs can cause submission errors**, especially from:
- Open3D 3D visualizations
- Large matplotlib plots
- Interactive widgets
- Large data displays

After clearing, you can safely use `ask_assist()` or `submit_eval()`.
    

We will next look at top 5 classes the network indicates to see if chair appears in any of them

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
points = np.asarray(cluster_pcds[6].points)
points = points[:,[0,1,2]]
result,probabilities = classify_point_cloud(points)
print(result)
top5_indices = probabilities.argsort()[-5:][::-1]

print("Top 5 Predictions:")
for idx in top5_indices:
    name = MODELNET40_CLASSES[idx]
    confidence = probabilities[idx] * 100
    print(f"{name:15} : {confidence:.2f}%")

---
**Q6**

Why does the chair not show up as the top class?

What steps could you take to try and improve the classification result for this chair? Think about both the data and the model.?


## Ans

This type of chair has an unusual shape -- you can notice that it doesnt have four legs - but two curved supports. This type of chair is not present in the modelnet40 training data - which was used to train the base model. This is also called out-of-distribution data.

The point cloud data is noisy. This means that even dbscan - which clusters points into groups is also noisy. Hence any classification based on this noisy input could lead to more uncertainty.

In addition, the farthest point sampling starts with an initial random point and then constructs 1024 subset of points which covers the entire object. However it is possible that this subset misses out certain key features.

All these contribute to a machine learning based model to failt to get the correct class.


**Chat with TA**

Check the question and answer

In [ ]:
ta.show_teaching_assist_button(session,AI_TA_URL, 6, notebook_id, institution_id, term_id, course_id)

**FOR INSTRUCTOR**

Upload the rubric

In [ ]:
ta.upload_rubric(session,AI_TA_URL, notebook_id, course_id,term_id, institution_id)

Call to AI TAssistant failed with status code: 500
Error message: {"detail":"An internal error occurred: 'AddRubricRequest' object has no attribute 'rubric_name'"}
